# Lunar Lander: reward shaping and hidden reward hacking

A step-by-step walkthrough of the whole project. Run the cells in order in Google Colab
(Runtime type: CPU is fine). Training cells take about 20 to 25 minutes each.

## 1. Install

In [ ]:
!apt-get install -y swig
!pip install swig "gymnasium[box2d]" stable-baselines3 moviepy

In [ ]:
import glob
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import Video
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
print("all good")

## 2. Explore the game

The agent sees 8 numbers (position, height, sideways speed, up/down speed, tilt, spin, two leg contacts)
and chooses one of 4 actions (0 nothing, 1 left engine, 2 main engine, 3 right engine).

In [ ]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")
print("Actions:", env.action_space)
print("Observations:", env.observation_space)

obs, info = env.reset(seed=0)
print(obs)

plt.imshow(env.render())
plt.axis("off")
plt.show()

Fire one engine for 20 frames and watch how sideways speed (`obs[2]`) and spin (`obs[5]`) change. Try actions 1, 2 and 3.

In [ ]:
obs, info = env.reset(seed=0)
print("before:", obs[2], obs[5])
for i in range(20):
    obs, reward, terminated, truncated, info = env.step(3)
print("after: ", obs[2], obs[5], "| tilt:", obs[4])
plt.imshow(env.render())
plt.axis("off")
plt.show()

## 3. Random baseline

In [ ]:
env = gym.make("LunarLander-v3")
scores = []
for episode in range(10):
    obs, info = env.reset(seed=episode)
    total, done = 0, False
    while not done:
        obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
        total += reward
        done = terminated or truncated
    scores.append(round(float(total), 1))
env.close()
print("Scores:", scores)
print("Average:", sum(scores) / len(scores))

## 4. Reward wrappers and test tools

In [ ]:
"""Reward wrappers used in the Lunar Lander reward-shaping experiments.

Each wrapper sits around the LunarLander-v3 environment and changes only the
reward. Observations, actions and episode endings pass through unchanged.

Observation indices (LunarLander-v3):
    0 x position, 1 height, 2 sideways speed, 3 up/down speed,
    4 tilt, 5 spin, 6 left leg touching, 7 right leg touching
"""

import gymnasium as gym


class TimePenalty(gym.Wrapper):
    """Charges a small fee on every frame, so dawdling costs points."""

    def __init__(self, env, fee=0.05):
        super().__init__(env)
        self.fee = fee

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        return obs, reward - self.fee, terminated, truncated, info


def in_hover_zone(obs, threshold=0.1):
    """True when the lander is upright and slow (the hover-bonus condition)."""
    return (abs(obs[4]) < threshold
            and abs(obs[2]) < threshold
            and abs(obs[3]) < threshold)


class HoverBonus(gym.Wrapper):
    """State-based bonus: pays +bonus on every frame the lander is upright and slow.

    Because it pays again on every frame the lander stays in the zone, it can
    be farmed. In our pilot the agent landed, then pulsed both side engines so
    the episode never ended, collecting the bonus until the time limit.
    """

    def __init__(self, env, bonus=1.0, threshold=0.1):
        super().__init__(env)
        self.bonus = bonus
        self.threshold = threshold

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        if in_hover_zone(obs, self.threshold):
            reward = reward + self.bonus
        return obs, reward, terminated, truncated, info


class ImprovementBonus(gym.Wrapper):
    """Potential-based bonus: pays for getting better, not for being good.

    reward += scale * (potential(now) - potential(previous frame))

    Summed over an episode this equals scale * (potential(end) - potential(start)),
    so staying still earns nothing and the bonus cannot be farmed.

    Note: the textbook form (Ng, Harada and Russell, 1999) uses
    gamma * potential(now) - potential(previous). We use gamma = 1 here.
    """

    def __init__(self, env, scale=10.0):
        super().__init__(env)
        self.scale = scale
        self.last = 0.0

    @staticmethod
    def potential(obs):
        return -(abs(obs[4]) + abs(obs[2]) + abs(obs[3]))

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.last = self.potential(obs)
        return obs, info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        now = self.potential(obs)
        reward = reward + self.scale * (now - self.last)
        self.last = now
        return obs, reward, terminated, truncated, info


def make_wrapper(reward):
    """Returns the wrapper function for one of the named reward conditions."""
    if reward == "default":
        return None
    if reward == "fee":
        return lambda e: TimePenalty(e)
    if reward == "hover":
        return lambda e: HoverBonus(TimePenalty(e))
    if reward == "improve":
        return lambda e: ImprovementBonus(TimePenalty(e))
    raise ValueError(f"Unknown reward condition: {reward}")

In [ ]:
"""Evaluate trained agents on the plain LunarLander-v3 game.

Every agent is judged on the real game score, with no fee or bonus, on the
same held-out starts (seeds 100 onward). Each test runs twice:

    dice off  deterministic=True   always press the most likely action
    dice on   deterministic=False  sample from the action probabilities

Examples:
    python evaluate.py models/ppo_hover_seed0.zip
    python evaluate.py models/*.zip --episodes 100
"""

import argparse
import glob

import gymnasium as gym
from stable_baselines3 import PPO



def evaluate(model, n=20, det=True, first_seed=100, verbose=True):
    """Plays n test episodes and reports score, length and time-limit count."""
    env = gym.make("LunarLander-v3")
    scores, lengths, time_limits, ground_stalls = [], [], 0, 0
    for episode in range(n):
        obs, info = env.reset(seed=first_seed + episode)
        total, frames, done = 0.0, 0, False
        while not done:
            action, _ = model.predict(obs, deterministic=det)
            obs, reward, terminated, truncated, info = env.step(action)
            total += reward
            frames += 1
            done = terminated or truncated
        scores.append(round(float(total), 1))
        lengths.append(frames)
        if truncated:
            time_limits += 1
            if obs[6] == 1 and obs[7] == 1:
                ground_stalls += 1
    env.close()
    result = {
        "mean_score": round(sum(scores) / n, 1),
        "mean_length": round(sum(lengths) / n),
        "time_limits": time_limits,
        "ground_stalls": ground_stalls,
        "episodes": n,
        "scores": scores,
        "lengths": lengths,
    }
    if verbose:
        print(f"  Average score:  {result['mean_score']}")
        print(f"  Average length: {result['mean_length']}")
        print(f"  Hit time limit: {time_limits} out of {n}"
              f" ({ground_stalls} with both legs down)")
    return result


def inspect_episode(model, seed=100, det=False):
    """Plays one episode and counts frames in the hover zone and on the ground."""
    env = gym.make("LunarLander-v3")
    obs, info = env.reset(seed=seed)
    frames, in_zone, on_ground, done = 0, 0, 0, False
    while not done:
        action, _ = model.predict(obs, deterministic=det)
        obs, reward, terminated, truncated, info = env.step(action)
        frames += 1
        in_zone += in_hover_zone(obs)
        on_ground += int(obs[6] == 1 and obs[7] == 1)
        done = terminated or truncated
    env.close()
    print(f"Frames: {frames} | In bonus zone: {in_zone} | Both legs down: {on_ground}")
    return frames, in_zone, on_ground


def record_video(model, folder, seed=0, det=True):
    """Records one episode as an .mp4 in the given folder (needs moviepy)."""
    from gymnasium.wrappers import RecordVideo
    env = gym.make("LunarLander-v3", render_mode="rgb_array")
    env = RecordVideo(env, folder, episode_trigger=lambda e: True)
    obs, info = env.reset(seed=seed)
    total, frames, done = 0.0, 0, False
    while not done:
        action, _ = model.predict(obs, deterministic=det)
        obs, reward, terminated, truncated, info = env.step(action)
        total += reward
        frames += 1
        done = terminated or truncated
    env.close()
    print(f"Score: {total:.1f} | Frames: {frames} | "
          f"Ended by: {'landing/crash' if terminated else 'time limit'}")

## 5. Train the four agents

Each cell trains one agent for 1,000,000 timesteps and saves it. Download the `.zip` files from the
Files panel afterwards, because Colab deletes them when the session ends.

In [ ]:
def train(reward, seed=0, steps=1_000_000):
    env = make_vec_env("LunarLander-v3", n_envs=16, seed=seed,
                       wrapper_class=make_wrapper(reward))
    model = PPO("MlpPolicy", env,
                n_steps=1024, batch_size=64, n_epochs=4,
                gamma=0.999, gae_lambda=0.98, ent_coef=0.01,
                seed=seed, verbose=1)
    model.learn(total_timesteps=steps)
    model.save(f"ppo_{reward}_seed{seed}")
    return model

In [ ]:
model1 = train("default")   # Run 1: default game reward

In [ ]:
model2 = train("fee")       # Run 2: fee of 0.05 per frame

In [ ]:
model3 = train("hover")     # Run 3: fee + hover bonus

In [ ]:
model4 = train("improve")   # Run 4: fee + improvement bonus

## 6. Test with dice off and dice on

Dice off (`deterministic=True`) always presses the most likely action. Dice on (`deterministic=False`)
samples, the same way the agent behaved in training. Watch the "Hit time limit" line for Run 3.

In [ ]:
for name, m in [("Run 1 (default)", model1), ("Run 2 (fee)", model2),
                ("Run 3 (fee + hover bonus)", model3), ("Run 4 (fee + improvement bonus)", model4)]:
    print(f"\n=== {name} ===")
    print("Dice OFF:")
    evaluate(m, det=True)
    print("Dice ON:")
    evaluate(m, det=False)

## 7. Look at the hack

In [ ]:
inspect_episode(model3, seed=100, det=False)
record_video(model3, "hack_run", seed=100, det=False)
Video(glob.glob("hack_run/*.mp4")[0], embed=True)

## 8. Check that the improvement bonus cannot be farmed

In [ ]:
plain = gym.make("LunarLander-v3")
wrapped = ImprovementBonus(gym.make("LunarLander-v3"), scale=10.0)
obs, _ = plain.reset(seed=0)
obs_w, _ = wrapped.reset(seed=0)
start = wrapped.potential(obs_w)

total_extra, done = 0.0, False
while not done:
    action, _ = model2.predict(obs, deterministic=True)
    obs, r1, terminated, truncated, _ = plain.step(action)
    obs_w, r2, _, _, _ = wrapped.step(action)
    total_extra += r2 - r1
    done = terminated or truncated

end = wrapped.potential(obs_w)
print("Total bonus added:  ", round(total_extra, 4))
print("10 x (end - start): ", round(10 * (end - start), 4))